In [1]:

import pandas as pd
import os
import numpy as np

def load_csv_to_dataframe(file_path):
    """
    Load data from a CSV file into a Pandas DataFrame.

    Parameters:
    file_path (str): The path to the CSV file.

    Returns:
    pd.DataFrame: A DataFrame containing the data from the CSV file.
    """
    try:
        df = pd.read_csv(file_path)
        return df
    except Exception as e:
        print(f"Error loading CSV file: {e}")
        return None


# Example usage
file_path = r"C:\Users\timur\Documents\GitHub\EmailSentin\data\local_data\email_enron.csv"
dataframe = load_csv_to_dataframe(file_path)

if dataframe is not None:
    print(dataframe.head())  # Display the first few rows of the DataFrame




def separate_email_fields(df, column_name):
    def extract_fields(email_text):
        fields = {
            'From': None,
            'To': None,
            'Cc': None,
            'Bcc': None,
            'Date': None,
            'Subject': None,
            'Body': None
        }
        
        for line in email_text.split('\n'):
            line = line.strip()
            if line.startswith("From:"):
                fields['From'] = line[6:].strip()
            elif line.startswith("To:"):
                fields['To'] = line[4:].strip()
            elif line.startswith("Cc:"):
                fields['Cc'] = line[4:].strip()
            elif line.startswith("Bcc:"):
                fields['Bcc'] = line[5:].strip()
            elif line.startswith("Date:"):
                fields['Date'] = line[6:].strip()
            elif line.startswith("Subject:"):
                fields['Subject'] = line[9:].strip()
            elif line == "":
                break
        
        body_start = email_text.find('\n\n')
        if body_start != -1:
            fields['Body'] = email_text[body_start + 2:].strip()
        
        return fields

    separated_data = df[column_name].apply(extract_fields)
    separated_df = pd.DataFrame(list(separated_data))
    
    return separated_df


def irregular_union_indices(df, body_column):
    forwarded_pattern = r'(Forwarded message|Fwd:|Forwarded)'
    irregular_pattern = r'(\b(?:\d{1,3}(?:,\d{3})*|\d+)\s*(?:\t|,|;)\s*(?:\d{1,3}(?:,\d{3})*|\d+)|\|)'

    forwarded_indices = df.index[df[body_column].str.contains(forwarded_pattern, case=False, na=False)].tolist()
    irregular_indices = df.index[df[body_column].str.contains(irregular_pattern, case=False, na=False)].tolist()
    
    union_indices = set(forwarded_indices) | set(irregular_indices)
    opposite_indices = [i for i in df.index if i not in union_indices]
    
    return opposite_indices


def partition_and_save(df, num_partitions, base_path):
    df_shuffled = df.sample(frac=1).reset_index(drop=True)
    partition_size = len(df) // num_partitions
    
    for i in range(num_partitions):
        start_idx = i * partition_size
        end_idx = len(df) if i == num_partitions - 1 else (i + 1) * partition_size
        df_partition = df_shuffled.iloc[start_idx:end_idx]
        partition_file_path = os.path.join(base_path, f"email_enron_processed_partition_{i+1}.csv")
        df_partition.to_csv(partition_file_path, index=False)
        print(f"Partition {i+1} written to {partition_file_path}")


                       file                                            message
0     allen-p/_sent_mail/1.  Message-ID: <18782981.1075855378110.JavaMail.e...
1    allen-p/_sent_mail/10.  Message-ID: <15464986.1075855378456.JavaMail.e...
2   allen-p/_sent_mail/100.  Message-ID: <24216240.1075855687451.JavaMail.e...
3  allen-p/_sent_mail/1000.  Message-ID: <13505866.1075863688222.JavaMail.e...
4  allen-p/_sent_mail/1001.  Message-ID: <30922949.1075863688243.JavaMail.e...


In [2]:
separated_emails_df = separate_email_fields(dataframe, 'message')

In [3]:
indices_to_include = irregular_union_indices(separated_emails_df, 'Body')

C:\Users\timur\AppData\Local\Temp\ipykernel_37976\567616741.py:78: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  forwarded_indices = df.index[df[body_column].str.contains(forwarded_pattern, case=False, na=False)].tolist()
C:\Users\timur\AppData\Local\Temp\ipykernel_37976\567616741.py:79: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  irregular_indices = df.index[df[body_column].str.contains(irregular_pattern, case=False, na=False)].tolist()


In [4]:
ready_emails = separated_emails_df.iloc[indices_to_include]['Body'].reset_index(drop=True)

In [5]:
type(ready_emails)

pandas.core.series.Series

In [6]:
data_frame_to_write = pd.DataFrame({
    'mail': ready_emails,
    'formality_score': pd.Series(dtype='int'),  # Empty integer column
    'formality_binary': pd.Series(dtype='object')       # Empty object column
})

In [7]:
data_frame_to_write

,mail,formality_score,formality_binary
0,Here is our forecast,NaN,NaN
1,Traveling to have a business meeting takes the...,NaN,NaN
2,test successful. way to go!!!,NaN,NaN
3,"Randy,\n\n Can you send me a schedule of the s...",NaN,NaN
4,Let's shoot for Tuesday at 11:45.,NaN,NaN
...,...,...,...
281006,I am interested in a program for myself. I am...,NaN,NaN
281007,Here is the update list that you requested. Mi...,NaN,NaN
281008,Please set up access for the digital certifica...,NaN,NaN
281009,Some of my position is with the Alberta Term b...,NaN,NaN


In [8]:
data_frame_to_write

,mail,formality_score,formality_binary
0,Here is our forecast,NaN,NaN
1,Traveling to have a business meeting takes the...,NaN,NaN
2,test successful. way to go!!!,NaN,NaN
3,"Randy,\n\n Can you send me a schedule of the s...",NaN,NaN
4,Let's shoot for Tuesday at 11:45.,NaN,NaN
...,...,...,...
281006,I am interested in a program for myself. I am...,NaN,NaN
281007,Here is the update list that you requested. Mi...,NaN,NaN
281008,Please set up access for the digital certifica...,NaN,NaN
281009,Some of my position is with the Alberta Term b...,NaN,NaN


In [10]:

base_output_path = r"C:\Users\timur\Documents\GitHub\EmailSentin\data\local_data\partitions"

num_partitions = 10

partition_and_save(data_frame_to_write, num_partitions, base_output_path)



Partition 1 written to C:\Users\timur\Documents\GitHub\EmailSentin\data\local_data\partitions\email_enron_processed_partition_1.csv
Partition 2 written to C:\Users\timur\Documents\GitHub\EmailSentin\data\local_data\partitions\email_enron_processed_partition_2.csv
Partition 3 written to C:\Users\timur\Documents\GitHub\EmailSentin\data\local_data\partitions\email_enron_processed_partition_3.csv
Partition 4 written to C:\Users\timur\Documents\GitHub\EmailSentin\data\local_data\partitions\email_enron_processed_partition_4.csv
Partition 5 written to C:\Users\timur\Documents\GitHub\EmailSentin\data\local_data\partitions\email_enron_processed_partition_5.csv
Partition 6 written to C:\Users\timur\Documents\GitHub\EmailSentin\data\local_data\partitions\email_enron_processed_partition_6.csv
Partition 7 written to C:\Users\timur\Documents\GitHub\EmailSentin\data\local_data\partitions\email_enron_processed_partition_7.csv
Partition 8 written to C:\Users\timur\Documents\GitHub\EmailSentin\data\loca